# 🛠️ Preprocessing Data — Evalify Dashboard Karir
> **Capstone CC26-PSU095 · Coding Camp 2026 powered by DBS Foundation**

Notebook ini memproses semua dataset mentah yang digunakan dashboard Streamlit Evalify
menjadi format bersih yang siap pakai. Jalankan tiap section secara berurutan.

## Dataset yang diproses

| # | File Mentah | File Output | Digunakan di Halaman |
|---|-------------|-------------|----------------------|
| 1 | `pengangguran_bps_1986_2024.csv` | `bps_pengangguran_clean.csv` | Pasar Kerja ID, Time Series |
| 2 | `ump_df.csv` + `ump_2026.csv` | `ump_lengkap_clean.csv` | Pasar Kerja ID, Time Series |
| 3 | `tpt_provinsi_2026.csv` | `tpt_clean.csv` | Pasar Kerja ID |
| 4 | `upah_df.csv` | `upah_aktual_clean.csv` | Pasar Kerja ID |
| 5 | `Job_Description_and_Salary_in_Indonesia.csv` | `loker_indonesia_clean.csv` | Pasar Kerja ID |
| 6 | `postings.csv` + `job_industries.csv` + `industries.csv` | `postings_clean.csv`, `postings_industri_clean.csv` | Rekrutmen & Industri |
| 7 | `job_skills.csv` + `skills.csv` | `skill_demand_linkedin_clean.csv` | Rekrutmen & Industri |
| 8 | `jd_cleaned.csv` | `jd_preprocessed.csv` | Rekrutmen & Industri |

---


## 0. Setup & Import

In [ ]:
import os, re, ast, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

# Sesuaikan path ke folder data jika perlu
DATA_DIR   = "data"           # folder data mentah
OUTPUT_DIR = "data/processed" # folder output hasil preprocessing

os.makedirs(OUTPUT_DIR, exist_ok=True)

def _p(f):  return os.path.join(DATA_DIR, f)
def _o(f):  return os.path.join(OUTPUT_DIR, f)

print("✅ Setup selesai")
print(f"   Data dir   : {os.path.abspath(DATA_DIR)}")
print(f"   Output dir : {os.path.abspath(OUTPUT_DIR)}")


## 1. Helper — Normalisasi Nama Provinsi
Digunakan di semua dataset Indonesia agar nama provinsi konsisten.

In [ ]:
# Mapping nama provinsi dari berbagai format BPS ke nama standar
PROVINSI_NORM = {
    # Aceh
    "ACEH": "Aceh", "NAD": "Aceh", "Nanggroe Aceh Darussalam": "Aceh",
    # Sumatera
    "SUMATERA UTARA": "Sumatera Utara", "SUMUT": "Sumatera Utara",
    "SUMATERA BARAT": "Sumatera Barat", "SUMBAR": "Sumatera Barat",
    "SUMATERA SELATAN": "Sumatera Selatan", "SUMSEL": "Sumatera Selatan",
    "RIAU": "Riau", "KEPULAUAN RIAU": "Kepulauan Riau", "KEPRI": "Kepulauan Riau",
    "JAMBI": "Jambi",
    "BENGKULU": "Bengkulu",
    "LAMPUNG": "Lampung",
    "KEPULAUAN BANGKA BELITUNG": "Kep. Bangka Belitung",
    "BANGKA BELITUNG": "Kep. Bangka Belitung", "BABEL": "Kep. Bangka Belitung",
    # Jawa
    "DKI JAKARTA": "DKI Jakarta", "JAKARTA": "DKI Jakarta",
    "JAWA BARAT": "Jawa Barat", "JABAR": "Jawa Barat",
    "JAWA TENGAH": "Jawa Tengah", "JATENG": "Jawa Tengah",
    "JAWA TIMUR": "Jawa Timur", "JATIM": "Jawa Timur",
    "DI YOGYAKARTA": "DI Yogyakarta", "DIY": "DI Yogyakarta", "D.I. YOGYAKARTA": "DI Yogyakarta",
    "BANTEN": "Banten",
    # Kalimantan
    "KALIMANTAN BARAT": "Kalimantan Barat", "KALBAR": "Kalimantan Barat",
    "KALIMANTAN TENGAH": "Kalimantan Tengah", "KALTENG": "Kalimantan Tengah",
    "KALIMANTAN SELATAN": "Kalimantan Selatan", "KALSEL": "Kalimantan Selatan",
    "KALIMANTAN TIMUR": "Kalimantan Timur", "KALTIM": "Kalimantan Timur",
    "KALIMANTAN UTARA": "Kalimantan Utara", "KALTARA": "Kalimantan Utara",
    # Sulawesi
    "SULAWESI UTARA": "Sulawesi Utara", "SULUT": "Sulawesi Utara",
    "SULAWESI TENGAH": "Sulawesi Tengah", "SULTENG": "Sulawesi Tengah",
    "SULAWESI SELATAN": "Sulawesi Selatan", "SULSEL": "Sulawesi Selatan",
    "SULAWESI TENGGARA": "Sulawesi Tenggara", "SULTRA": "Sulawesi Tenggara",
    "GORONTALO": "Gorontalo",
    "SULAWESI BARAT": "Sulawesi Barat", "SULBAR": "Sulawesi Barat",
    # Bali & Nusa Tenggara
    "BALI": "Bali",
    "NUSA TENGGARA BARAT": "Nusa Tenggara Barat", "NTB": "Nusa Tenggara Barat",
    "NUSA TENGGARA TIMUR": "Nusa Tenggara Timur", "NTT": "Nusa Tenggara Timur",
    # Maluku & Papua
    "MALUKU": "Maluku",
    "MALUKU UTARA": "Maluku Utara",
    "PAPUA": "Papua",
    "PAPUA BARAT": "Papua Barat",
    "PAPUA SELATAN": "Papua Selatan",
    "PAPUA TENGAH": "Papua Tengah",
    "PAPUA PEGUNUNGAN": "Papua Pegunungan",
    "PAPUA BARAT DAYA": "Papua Barat Daya",
}

def norm_prov(name):
    """Normalisasi nama provinsi ke format standar."""
    s = str(name).strip()
    return PROVINSI_NORM.get(s, PROVINSI_NORM.get(s.upper(), s.title()))

print("✅ Mapping provinsi siap:", len(PROVINSI_NORM), "entri")


---
## 2. BPS Pengangguran 1986–2024
**File mentah:** `pengangguran_bps_1986_2024.csv`  
**Masalah:** Format wide dengan multi-level header (row 1–2 = judul/keterangan bulan), kolom campuran satu nilai (1986–2004) dan dua nilai Feb/Nov per tahun (2005–2024). BOM UTF-8, banyak sel kosong.  
**Output:** `bps_pengangguran_clean.csv` — format long (Tahun, Pendidikan, Jumlah)


In [ ]:
fp_bps = _p("pengangguran_bps_1986_2024.csv")
raw    = pd.read_csv(fp_bps, header=None, skiprows=2, low_memory=False)

# Hanya baris yang kolom-0 nya angka (No. baris data)
mask = pd.to_numeric(raw.iloc[:, 0], errors="coerce").notna()
edu  = raw[mask].reset_index(drop=True).iloc[:8]

print("Raw edu block (8 rows × pertama 5 col):")
print(edu.iloc[:, :5])
print(f"\nTotal kolom: {edu.shape[1]}")


In [ ]:
# Mapping label pendidikan BPS → nama bersih
MAP_EDU = {
    "Tidak/belum pernah sekolah": "Tidak Sekolah",
    "Tidak/belum tamat SD":       "Tidak Tamat SD",
    "SD":                         "SD",
    "SLTP":                       "SMP",
    "SLTA Umum/SMU":              "SMA",
    "SLTA Kejuruan/SMK":          "SMK",
    "Akademi/Diploma":            "Diploma / D3",
    "Universitas":                "Universitas / S1+",
}

levels = edu.iloc[:, 1].tolist()
rows   = []

# Kolom 2–20 → 1986–2004 (satu nilai per tahun, tidak ada pemisahan Feb/Nov)
for off, yr in enumerate(range(1986, 2005)):
    col = 2 + off
    for i, lv in enumerate(levels):
        v = edu.iloc[i, col]
        if pd.notna(v):
            try:
                rows.append({
                    "Tahun":      yr,
                    "Pendidikan": MAP_EDU.get(lv, lv),
                    "Jumlah":     float(str(v).replace(",", ".")),
                    "Periode":    "Tahunan",
                })
            except Exception:
                pass

# Kolom 21+ → 2005–2024 (dua kolom per tahun: Feb & Agustus/Nov)
# Kita ambil kolom genap (Februari) sebagai representasi tahunan
for yo in range(20):
    yr  = 2005 + yo
    col = 21 + yo * 2   # kolom Februari
    for i, lv in enumerate(levels):
        try:
            v = edu.iloc[i, col]
            if pd.notna(v):
                rows.append({
                    "Tahun":      yr,
                    "Pendidikan": MAP_EDU.get(lv, lv),
                    "Jumlah":     float(str(v).replace(",", ".")),
                    "Periode":    "Februari",
                })
        except Exception:
            pass

df_bps = pd.DataFrame(rows)

# Validasi
assert df_bps["Tahun"].nunique() == 39, "Jumlah tahun tidak sesuai"
assert df_bps["Pendidikan"].nunique() == 8, "Jumlah kategori pendidikan tidak sesuai"
assert df_bps["Jumlah"].isna().sum() == 0, "Ada nilai NaN"

print(f"Shape: {df_bps.shape}")
print(f"Tahun: {df_bps['Tahun'].min()} – {df_bps['Tahun'].max()}")
print(f"Kategori pendidikan: {df_bps['Pendidikan'].unique().tolist()}")
df_bps.head(10)


In [ ]:
# Simpan output
df_bps.to_csv(_o("bps_pengangguran_clean.csv"), index=False)
print("✅ Disimpan:", _o("bps_pengangguran_clean.csv"))

# Preview pivot untuk Time Series
pivot_bps = df_bps.pivot_table(index="Tahun", columns="Pendidikan", values="Jumlah").reset_index()
pivot_bps.to_csv(_o("bps_pengangguran_pivot.csv"), index=False)
print("✅ Disimpan:", _o("bps_pengangguran_pivot.csv"))
print("\nPivot (5 baris terakhir):")
pivot_bps.tail()


---
## 3. UMP per Provinsi 2002–2026
**File mentah:**  
- `ump_df.csv` — data historis 2002–2025, format long, nama provinsi UPPERCASE  
- `ump_2026.csv` — data 2026, format wide dengan 2 baris header BPS  

**Masalah:** Dua sumber berbeda format, nama provinsi tidak konsisten, ada baris total/header yang ikut terbaca, nilai non-numerik.  
**Output:** `ump_lengkap_clean.csv` — format long (Provinsi, Tahun, UMP, UMP_Juta)


In [ ]:
# --- ump_df.csv (2002-2025) ---
h = pd.read_csv(_p("ump_df.csv"))
print("ump_df shape:", h.shape)
print(h.head())
print("\nCek null:", h.isnull().sum().to_dict())
print("Cek duplikat:", h.duplicated().sum())


In [ ]:
# --- ump_2026.csv (format mentah BPS) ---
d26_raw = pd.read_csv(_p("ump_2026.csv"), header=None, skiprows=2,
                       names=["provinsi", "ump"])
print("ump_2026 raw (10 baris):")
print(d26_raw.head(10))


In [ ]:
# Bersihkan ump_2026.csv
d26 = d26_raw.copy()
d26["ump"]  = pd.to_numeric(d26["ump"], errors="coerce")
d26 = d26[d26["ump"].notna()].copy()           # buang baris header/kosong
d26 = d26[d26["ump"] > 1_000_000].copy()       # buang nilai tak wajar (< 1 juta)
d26["tahun"] = 2026
print(f"ump_2026 setelah cleaning: {d26.shape}")
print(d26.head())


In [ ]:
# Gabung & normalisasi provinsi
h_sel  = h[["provinsi", "tahun", "ump"]].copy()
d26_sel = d26[["provinsi", "tahun", "ump"]].copy()

ump_all = pd.concat([h_sel, d26_sel], ignore_index=True)
ump_all.columns = ["Provinsi", "Tahun", "UMP"]

# Normalisasi nama provinsi
ump_all["Provinsi"] = ump_all["Provinsi"].apply(norm_prov)

# Filter baris yang tidak masuk akal
ump_all = ump_all[ump_all["UMP"] > 100_000].copy()

# Deduplikat: kalau ada provinsi+tahun ganda, ambil yang terbesar
ump_all = (ump_all.sort_values("UMP", ascending=False)
                  .drop_duplicates(subset=["Provinsi", "Tahun"])
                  .sort_values(["Provinsi", "Tahun"])
                  .reset_index(drop=True))

# Tambah kolom jutaan rupiah
ump_all["UMP_Juta"] = (ump_all["UMP"] / 1e6).round(3)

print(f"Shape final: {ump_all.shape}")
print(f"Tahun: {ump_all['Tahun'].min()} – {ump_all['Tahun'].max()}")
print(f"Provinsi unik: {ump_all['Provinsi'].nunique()}")
ump_all.head(10)


In [ ]:
# Cek provinsi yang masuk
print("Daftar provinsi:")
for p in sorted(ump_all["Provinsi"].unique()):
    print(" ", p)


In [ ]:
# Simpan
ump_all.to_csv(_o("ump_lengkap_clean.csv"), index=False)
print("✅ Disimpan:", _o("ump_lengkap_clean.csv"))

# Pivot untuk Time Series (baris=tahun, kolom=provinsi, nilai=UMP_Juta)
ump_pivot = ump_all.pivot_table(index="Tahun", columns="Provinsi", values="UMP_Juta")
ump_pivot = ump_pivot.ffill()   # forward-fill gap data
ump_pivot.to_csv(_o("ump_pivot_clean.csv"))
print("✅ Disimpan:", _o("ump_pivot_clean.csv"))
ump_pivot.tail()


---
## 4. TPT per Provinsi 2026
**File mentah:** `tpt_provinsi_2026.csv`  
**Masalah:** 3 baris header BPS, nama provinsi UPPERCASE, kolom Agustus & Tahunan bertanda "-" (kosong), ada baris total Indonesia.  
**Output:** `tpt_clean.csv` — (Provinsi, TPT_Feb2026)


In [ ]:
fp_tpt = _p("tpt_provinsi_2026.csv")
tpt_raw = pd.read_csv(fp_tpt, header=None, skiprows=3,
                       names=["prov_raw", "TPT_Feb2026", "Agustus", "Tahunan"])
print("Raw TPT (10 baris):")
print(tpt_raw.head(10))


In [ ]:
# Cleaning
df_tpt = tpt_raw.copy()
df_tpt["TPT_Feb2026"] = pd.to_numeric(df_tpt["TPT_Feb2026"], errors="coerce")
df_tpt = df_tpt[df_tpt["TPT_Feb2026"].notna()].copy()

# Normalisasi nama
df_tpt["Provinsi"] = df_tpt["prov_raw"].apply(norm_prov)

# Pisahkan baris INDONESIA (nasional) & provinsi
df_tpt_nas  = df_tpt[df_tpt["prov_raw"].str.upper().str.contains("INDONESIA|TOTAL", na=False)]
df_tpt_prov = df_tpt[~df_tpt["prov_raw"].str.upper().str.contains("INDONESIA|TOTAL", na=False)]

print(f"Baris nasional: {len(df_tpt_nas)}")
print(f"Baris provinsi: {len(df_tpt_prov)}")
print(f"\nNilai TPT nasional: {df_tpt_nas['TPT_Feb2026'].values}")
df_tpt_prov[["Provinsi", "TPT_Feb2026"]].head(10)


In [ ]:
# Validasi range TPT (persen, wajar 1–30%)
assert df_tpt_prov["TPT_Feb2026"].between(0, 30).all(), "Ada nilai TPT di luar range wajar"
print(f"TPT min: {df_tpt_prov['TPT_Feb2026'].min():.2f}%")
print(f"TPT max: {df_tpt_prov['TPT_Feb2026'].max():.2f}%")

# Simpan
df_tpt_prov[["Provinsi", "prov_raw", "TPT_Feb2026"]].to_csv(_o("tpt_clean.csv"), index=False)
print("\n✅ Disimpan:", _o("tpt_clean.csv"))


---
## 5. Upah Aktual per Provinsi (upah_df.csv)
**File mentah:** `upah_df.csv`  
**Masalah:** Kolom `upah` dalam satuan IDR/jam — perlu konversi ke IDR/bulan (×8 jam ×22 hari kerja). Nama provinsi UPPERCASE.  
**Output:** `upah_aktual_clean.csv`


In [ ]:
df_upah = pd.read_csv(_p("upah_df.csv"))
print("Shape:", df_upah.shape)
print(df_upah.head())
print("\nDescribe upah (IDR/jam):")
print(df_upah["upah"].describe())


In [ ]:
df_upah_clean = df_upah.copy()
df_upah_clean["Provinsi"]      = df_upah_clean["provinsi"].apply(norm_prov)
df_upah_clean["upah_per_jam"]  = pd.to_numeric(df_upah_clean["upah"], errors="coerce")

# Konversi: IDR/jam → IDR/bulan (8 jam/hari × 22 hari kerja)
df_upah_clean["upah_bulanan"]  = df_upah_clean["upah_per_jam"] * 8 * 22
df_upah_clean["Tahun"]         = df_upah_clean["tahun"]

# Buang outlier ekstrem (< UMR minimum atau > 100 juta/bulan)
sebelum = len(df_upah_clean)
df_upah_clean = df_upah_clean[
    df_upah_clean["upah_bulanan"].between(500_000, 100_000_000)
].copy()
print(f"Baris dibuang karena outlier: {sebelum - len(df_upah_clean)}")

df_out = df_upah_clean[["Provinsi", "Tahun", "upah_per_jam", "upah_bulanan"]].dropna()
print(f"\nShape final: {df_out.shape}")
print(df_out.head())


In [ ]:
df_out.to_csv(_o("upah_aktual_clean.csv"), index=False)
print("✅ Disimpan:", _o("upah_aktual_clean.csv"))


---
## 6. Loker Indonesia (Job Description & Salary)
**File mentah:** `Job_Description_and_Salary_in_Indonesia.csv`  
**Masalah:**  
- Separator `|` (bukan koma)  
- Kolom `salary` mayoritas kosong, perlu filter nilai masuk akal  
- `job_function` bisa berisi multiple value dipisah koma  
- Banyak teks bebas di `job_description` (tidak diproses di sini, hanya dibersihkan)  
**Output:** `loker_indonesia_clean.csv`


In [ ]:
df_lkr_raw = pd.read_csv(_p("Job_Description_and_Salary_in_Indonesia.csv"),
                         sep="|", low_memory=False)
print("Shape mentah:", df_lkr_raw.shape)
print("Kolom:", df_lkr_raw.columns.tolist())
print("\nNull per kolom:")
print(df_lkr_raw.isnull().sum())


In [ ]:
df_lkr = df_lkr_raw.copy()

# 1. Bersihkan kolom salary
df_lkr["salary"] = pd.to_numeric(df_lkr["salary"], errors="coerce")

# Filter: hanya simpan baris dengan salary valid (500 rb – 500 jt IDR)
# dan baris tanpa salary (NaN dipertahankan karena tetap ada info lain)
mask_salary_ok = (
    df_lkr["salary"].isna() |
    df_lkr["salary"].between(500_000, 500_000_000)
)
df_lkr = df_lkr[mask_salary_ok].copy()
print(f"Baris setelah filter salary: {len(df_lkr):,}")
print(f"Baris dengan salary terisi: {df_lkr['salary'].notna().sum():,}")

# 2. Ekstrak fungsi pekerjaan pertama (sebelum koma pertama)
df_lkr["fungsi_singkat"] = (df_lkr["job_function"]
                             .str.split(",").str[0]
                             .str.strip()
                             .fillna("Lainnya"))

# 3. Bersihkan location (hapus spasi berlebih)
df_lkr["location"] = df_lkr["location"].str.strip()

# 4. Standarisasi career_level & employment_type
df_lkr["career_level"]    = df_lkr["career_level"].str.strip()
df_lkr["employment_type"] = df_lkr["employment_type"].str.strip()

# 5. Hapus duplikat berdasarkan id
df_lkr = df_lkr.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(f"Shape final: {df_lkr.shape}")
df_lkr[["job_title", "location", "career_level", "salary", "fungsi_singkat"]].head(5)


In [ ]:
# Statistik ringkas
print("Top 10 location:")
print(df_lkr["location"].value_counts().head(10))
print("\nTop 10 company_industry:")
print(df_lkr["company_industry"].value_counts().head(10))
print("\nTop 10 fungsi_singkat:")
print(df_lkr["fungsi_singkat"].value_counts().head(10))


In [ ]:
# Kolom yang disimpan (tanpa job_description yang sangat panjang)
COLS_OUT = ["id", "job_title", "location", "salary_currency", "career_level",
            "experience_level", "education_level", "employment_type",
            "job_function", "fungsi_singkat", "company_size",
            "company_industry", "salary"]

df_lkr[COLS_OUT].to_csv(_o("loker_indonesia_clean.csv"), index=False)
print("✅ Disimpan:", _o("loker_indonesia_clean.csv"))


---
## 7. LinkedIn Postings
**File mentah:** `postings.csv` (>123rb baris, ~30 kolom), `job_industries.csv`, `industries.csv`  
**Masalah:**  
- `listed_time` dalam format Unix milliseconds  
- Banyak kolom tidak diperlukan (description, url, dll)  
- Perlu merge dengan industries untuk mendapatkan nama industri  
**Output:** `postings_clean.csv`, `postings_industri_clean.csv`


In [ ]:
# Hanya muat kolom yang diperlukan dashboard
COLS_POST = [
    "job_id", "title", "formatted_experience_level",
    "formatted_work_type", "listed_time", "normalized_salary",
    "location", "remote_allowed"
]

df_post = pd.read_csv(_p("postings.csv"), usecols=COLS_POST, low_memory=False)
print("Shape postings:", df_post.shape)
print(df_post.dtypes)
print("\nNull per kolom:")
print(df_post.isnull().sum())


In [ ]:
df_post_clean = df_post.copy()

# 1. Konversi timestamp Unix ms → datetime
df_post_clean["listed_dt"] = pd.to_datetime(
    df_post_clean["listed_time"], unit="ms", errors="coerce"
)
df_post_clean["Bulan"]    = df_post_clean["listed_dt"].dt.to_period("M").astype(str)
df_post_clean["Kuartal"]  = df_post_clean["listed_dt"].dt.to_period("Q").astype(str)
df_post_clean["Tahun"]    = df_post_clean["listed_dt"].dt.year

# 2. Bersihkan title
df_post_clean["title_clean"] = df_post_clean["title"].str.strip().str.lower()

# 3. Standarisasi kolom kategori
df_post_clean["formatted_experience_level"] = df_post_clean["formatted_experience_level"].str.strip()
df_post_clean["formatted_work_type"]        = df_post_clean["formatted_work_type"].str.strip()

# 4. Hapus duplikat job_id
sebelum = len(df_post_clean)
df_post_clean = df_post_clean.drop_duplicates(subset=["job_id"]).reset_index(drop=True)
print(f"Duplikat dihapus: {sebelum - len(df_post_clean)}")
print(f"Shape final: {df_post_clean.shape}")

# Statistik
print(f"\nRentang waktu: {df_post_clean['listed_dt'].min()} – {df_post_clean['listed_dt'].max()}")
print(f"Experience level unik: {df_post_clean['formatted_experience_level'].dropna().unique()}")
print(f"Work type unik: {df_post_clean['formatted_work_type'].dropna().unique()}")


In [ ]:
# Merge dengan industri
df_ji  = pd.read_csv(_p("job_industries.csv"))
df_ind = pd.read_csv(_p("industries.csv"))

df_post_ind = (df_post_clean
               .merge(df_ji,  on="job_id",      how="left")
               .merge(df_ind, on="industry_id",  how="left"))

print("Shape postings + industri:", df_post_ind.shape)
print("Kolom:", df_post_ind.columns.tolist())
df_post_ind.head(3)


In [ ]:
# Simpan
COLS_POST_OUT = [
    "job_id", "title", "title_clean", "formatted_experience_level",
    "formatted_work_type", "listed_dt", "Bulan", "Kuartal", "Tahun",
    "normalized_salary", "location", "remote_allowed"
]
df_post_clean[COLS_POST_OUT].to_csv(_o("postings_clean.csv"), index=False)
print("✅ Disimpan:", _o("postings_clean.csv"))

COLS_IND_OUT = [
    "job_id", "title_clean", "industry_id", "industry_name",
    "formatted_experience_level", "formatted_work_type",
    "Bulan", "Kuartal", "Tahun", "location"
]
df_post_ind[COLS_IND_OUT].to_csv(_o("postings_industri_clean.csv"), index=False)
print("✅ Disimpan:", _o("postings_industri_clean.csv"))


---
## 8. LinkedIn Skill Demand
**File mentah:** `job_skills.csv` (mapping job_id → skill_abr), `skills.csv` (mapping skill_abr → nama)  
**Masalah:** Dua file terpisah perlu di-merge, lalu dihitung frekuensi per skill.  
**Output:** `skill_demand_linkedin_clean.csv` — (skill_abr, skill_name, Frekuensi)


In [ ]:
df_js = pd.read_csv(_p("job_skills.csv"))
df_sk = pd.read_csv(_p("skills.csv"))

print("job_skills shape:", df_js.shape)
print(df_js.head())
print("\nskills:")
print(df_sk)


In [ ]:
# Merge & hitung frekuensi
df_skill_merged = df_js.merge(df_sk, on="skill_abr", how="left")

# Validasi: tidak boleh ada skill_abr tanpa nama
missing = df_skill_merged["skill_name"].isna().sum()
print(f"Skill tanpa nama: {missing}")

# Hitung frekuensi
skill_demand = (df_skill_merged["skill_name"]
                .value_counts()
                .reset_index()
                .rename(columns={"skill_name": "skill_name", "count": "Frekuensi"}))
skill_demand.columns = ["skill_name", "Frekuensi"]

# Gabungkan kembali dengan skill_abr
skill_demand = skill_demand.merge(df_sk, on="skill_name", how="left")
skill_demand = skill_demand[["skill_abr", "skill_name", "Frekuensi"]]

print(f"\nShape: {skill_demand.shape}")
skill_demand.head(15)


In [ ]:
# Klasifikasi skill (sesuai logika dashboard)
TEKNIS = {
    "information technology", "engineering", "manufacturing",
    "health care provider", "project management", "analyst", "research",
    "quality assurance", "science", "production", "supply chain",
    "distribution", "purchasing", "design", "art/creative",
    "writing/editing", "product management", "advertising",
}
BISNIS = {
    "sales", "management", "business development", "other", "finance",
    "accounting/auditing", "administrative", "customer service",
    "human resources", "legal", "consulting", "education", "training",
    "marketing", "general business", "public relations", "strategy/planning",
}

skill_demand["Tipe"] = skill_demand["skill_name"].apply(
    lambda s: "Teknis & Spesialis"   if str(s).lower() in TEKNIS
         else ("Bisnis & Manajerial" if str(s).lower() in BISNIS else "Lainnya")
)

print("Distribusi tipe skill:")
print(skill_demand["Tipe"].value_counts())
skill_demand.head(10)


In [ ]:
skill_demand.to_csv(_o("skill_demand_linkedin_clean.csv"), index=False)
print("✅ Disimpan:", _o("skill_demand_linkedin_clean.csv"))


---
## 9. Job Description Cleaned (jd_cleaned.csv)
**File mentah:** `jd_cleaned.csv`  
**Catatan:** File ini sudah melalui preprocessing awal (cleaning teks). Preprocessing tambahan di sini:  
- Normalisasi `title_clean`  
- Parse `job_skills` dari string list Python → list proper  
- Statistik skill per posting  
**Output:** `jd_preprocessed.csv`


In [ ]:
df_jd = pd.read_csv(_p("jd_cleaned.csv"))
print("Shape:", df_jd.shape)
print("Kolom:", df_jd.columns.tolist())
print("\nNull:")
print(df_jd.isnull().sum())
df_jd.head(2)


In [ ]:
df_jd_clean = df_jd.copy()

# 1. Normalisasi title_clean
df_jd_clean["title_clean"] = df_jd_clean["title_clean"].str.strip().str.lower()

# 2. Parse job_skills dari string representasi list Python
def parse_skills(x):
    """Konversi string list Python ke list Python."""
    if pd.isna(x) or str(x).strip() in ("", "[]"):
        return []
    try:
        result = ast.literal_eval(str(x))
        return [s.strip().lower() for s in result if s.strip()]
    except Exception:
        return []

df_jd_clean["skills_list"]  = df_jd_clean["job_skills"].apply(parse_skills)
df_jd_clean["skill_count"]  = df_jd_clean["skills_list"].apply(len)

# 3. Hapus baris tanpa title atau description sama sekali
sebelum = len(df_jd_clean)
df_jd_clean = df_jd_clean[
    df_jd_clean["title_clean"].notna() &
    (df_jd_clean["title_clean"] != "")
].reset_index(drop=True)
print(f"Baris dihapus (title kosong): {sebelum - len(df_jd_clean)}")

# 4. Statistik skill
print(f"\nJumlah posting: {len(df_jd_clean):,}")
print(f"Skill count: min={df_jd_clean['skill_count'].min()}, "
      f"median={df_jd_clean['skill_count'].median():.0f}, "
      f"max={df_jd_clean['skill_count'].max()}")
print(f"Posting dengan skill: {(df_jd_clean['skill_count'] > 0).sum():,} "
      f"({(df_jd_clean['skill_count'] > 0).mean()*100:.1f}%)")

df_jd_clean[["title_clean", "skill_count", "skills_list"]].head(5)


In [ ]:
# Top skill dari jd_cleaned
from collections import Counter

all_skills = Counter()
for sk_list in df_jd_clean["skills_list"]:
    all_skills.update(sk_list)

print("Top 20 skill dari jd_cleaned:")
for sk, cnt in all_skills.most_common(20):
    print(f"  {sk:<40} {cnt:>6,}")


In [ ]:
# Simpan (tanpa kolom teks panjang agar lebih ringan)
COLS_JD = ["job_title", "title_clean", "job_skills", "skills_list", "skill_count",
           "text_combined"]
# simpan hanya kolom yang ada
COLS_JD_EXIST = [c for c in COLS_JD if c in df_jd_clean.columns]
df_jd_clean[COLS_JD_EXIST].to_csv(_o("jd_preprocessed.csv"), index=False)
print("✅ Disimpan:", _o("jd_preprocessed.csv"))


---
## 10. Ringkasan Output


In [ ]:
print("=" * 65)
print("RINGKASAN HASIL PREPROCESSING")
print("=" * 65)

output_files = [
    "bps_pengangguran_clean.csv",
    "bps_pengangguran_pivot.csv",
    "ump_lengkap_clean.csv",
    "ump_pivot_clean.csv",
    "tpt_clean.csv",
    "upah_aktual_clean.csv",
    "loker_indonesia_clean.csv",
    "postings_clean.csv",
    "postings_industri_clean.csv",
    "skill_demand_linkedin_clean.csv",
    "jd_preprocessed.csv",
]

total_size = 0
for fn in output_files:
    fp = _o(fn)
    if os.path.exists(fp):
        size_kb = os.path.getsize(fp) / 1024
        total_size += size_kb
        try:
            df_tmp = pd.read_csv(fp)
            print(f"  ✅ {fn:<45} {df_tmp.shape[0]:>8,} baris | {size_kb:>8.1f} KB")
        except Exception:
            print(f"  ✅ {fn:<45} (non-CSV atau gagal baca) | {size_kb:>8.1f} KB")
    else:
        print(f"  ❌ {fn:<45} TIDAK DITEMUKAN")

print("=" * 65)
print(f"  Total ukuran output: {total_size/1024:.1f} MB")
print(f"  Lokasi: {os.path.abspath(OUTPUT_DIR)}")


---
## Catatan Penggunaan di Dashboard

Setelah notebook ini dijalankan, file-file di `data/processed/` bisa langsung dipakai.  
Update `utils/loader.py` untuk mengarahkan ke path output baru:

```python
# Contoh perubahan di loader.py
OUTPUT_DIR = "data/processed"

@st.cache_data(show_spinner=False)
def pengangguran_bps():
    return pd.read_csv(os.path.join(OUTPUT_DIR, "bps_pengangguran_clean.csv"))

@st.cache_data(show_spinner=False)
def ump_lengkap():
    return pd.read_csv(os.path.join(OUTPUT_DIR, "ump_lengkap_clean.csv"))

# ... dst
```

> 💡 **Tips:** Jalankan notebook ini sekali, lalu simpan output ke `data/processed/`.  
> Dashboard akan jauh lebih cepat di-load karena tidak perlu parsing ulang format BPS yang kompleks setiap kali.
